In [ ]:
#SIMULATION - POLICY EFFECTIVENESS (Readme)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Module conducts the following analysis:

# ---- Figure 6: Differences in expected green H2 demand and policy cost effectiveness for a spatially targeted demand-side CfD
# ---- Supplementary Figure 15: Differences in expected green H2 demand and policy cost effectiveness for a spatially targeted demand-side CfD: Results for alternative baselines (Controlled setting)
# ---- Supplementary Figure 16: Differences in expected green H2 demand and policy cost effectiveness for a spatially targeted demand-side CfD: Results for alternative baselines (EHB setting)

# Module is input for:

# ---- NA

In [ ]:
#SET-UP
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
setwd("/home/h1604190/Spatially-informed Demand-side Policies for Green H2 Diffusion/") 
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.getenv("PROJ_LIB")

check_and_load <- function(packages) {
  for (pkg in packages) {
    if (!requireNamespace(pkg, quietly = TRUE)) {
      message(paste("Installing missing package:", pkg))
      install.packages(pkg, dependencies = TRUE, repos = "https://cloud.r-project.org")
    }
    if (!(pkg %in% (.packages()))) {
      suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    }
  }
}

# --- Required Libraries 
required_packages <- c(
  # Data handling
  "dplyr",
  "purrr",
  "data.table",
  "stringr",
  "tidyr",
  "tibble",
  "openxlsx",

  # Visualisation
  "ggplot2",
  "ggdist",
  "patchwork"
)

check_and_load(required_packages)


# Font
theme_set(
  theme_minimal(base_family = "Arial")
)

In [ ]:
#DATAFILES
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
results_all_dist <- readRDS("results_all_dist.rds")
results_all_dist_mean_restricted <- readRDS("results_all_dist_restricted_mean.rds")
results_all_dist_mean_extended <- readRDS("results_all_dist_extended_mean.rds")
results_all_dist_cons_central <- readRDS("results_all_dist_central_conservative.rds")
results_all_dist_cons_restricted <- readRDS("results_all_dist_restricted_conservative.rds")
results_all_dist_cons_extended <- readRDS("results_all_dist_extended_conservative.rds")
results_all_dist_progr_central <- readRDS("results_all_dist_central_progressive.rds")
results_all_dist_progr_restricted <- readRDS("results_all_dist_restricted_progressive.rds")
results_all_dist_progr_extended <- readRDS("results_all_dist_extended_progressive.rds")
results_all_dist_auction <- readRDS("results_all_dist_auction.rds")
results_all_dist_mean_restricted_auction <- readRDS("results_all_dist_restricted_mean_auction.rds")
results_all_dist_mean_extended_auction <- readRDS("results_all_dist_extended_mean_auction.rds")
results_all_dist_cons_central_auction <- readRDS("results_all_dist_central_conservative_auction.rds")
results_all_dist_cons_restricted_auction <- readRDS("results_all_dist_restricted_conservative_auction.rds")
results_all_dist_cons_extended_auction <- readRDS("results_all_dist_extended_conservative_auction.rds")
results_all_dist_progr_central_auction <- readRDS("results_all_dist_central_progressive_auction.rds")
results_all_dist_progr_restricted_auction <- readRDS("results_all_dist_restricted_progressive_auction.rds")
results_all_dist_progr_extended_auction <- readRDS("results_all_dist_extended_progressive_auction.rds")

In [ ]:
# FIGURE 6
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

plot_theme <- theme_minimal(base_size = 20) +
  theme(
    panel.grid       = element_blank(),
    panel.border     = element_rect(color = "black", fill = NA),
    axis.line        = element_line(color = "black"),
    axis.text        = element_text(size = 20),
    axis.title       = element_text(size = 20),
    strip.text       = element_text(size = 18, face = "plain"),
    strip.background = element_blank(),
    plot.title       = element_text(size = 20, face = "bold")
  )

extract_runs <- function(results_obj) {

  nm <- names(results_obj)
  out <- vector("list", length(nm))

  for (idx in seq_along(nm)) {

    name <- nm[[idx]]
    runs <- results_obj[[name]]

    dt_name <- rbindlist(
      lapply(seq_along(runs), function(i) {
        s <- runs[[i]]
        data.table(
          run_id        = i,
          year          = s$year,
          annual_demand = s$annual_demand / 1e3,
          annual_cost   = s$annual_cost,
          scenario_raw  = name
        )
      })
    )

    out[[idx]] <- dt_name
  }

  DT <- rbindlist(out)

  DT[, centrality_type := fifelse(
    grepl("betweenness", scenario_raw), "Betweenness",
    "Degree"
  )]

  DT[, intervention := fifelse(
    grepl("_high", scenario_raw), "High Centrality",
    fifelse(grepl("_low", scenario_raw), "Low Centrality",
    fifelse(grepl("_random|_cost", scenario_raw), "Random",
    "Baseline"))
  )]

  setorder(DT, run_id, centrality_type, intervention, year)

  DT[, cumulative_demand := cumsum(annual_demand),
     by = .(run_id, centrality_type, intervention)]
  DT[, cumulative_cost := cumsum(annual_cost),
     by = .(run_id, centrality_type, intervention)]

  baseDT <- DT[
    intervention == "Baseline",
    .(
      run_id,
      centrality_type,
      year,
      base_annual_demand = annual_demand,
      base_annual_cost   = annual_cost,
      base_cum_demand    = cumulative_demand,
      base_cum_cost      = cumulative_cost
    )
  ]

  M <- merge(
    DT[intervention != "Baseline"],
    baseDT,
    by = c("run_id", "centrality_type", "year"),
    all.x = TRUE
  )

  M[, additional_annual_demand := annual_demand - base_annual_demand]
  M[, additional_annual_cost   := annual_cost - base_annual_cost]

  M[, cumulative_additional_demand := cumulative_demand - base_cum_demand]
  M[, cumulative_additional_cost   := cumulative_cost - base_cum_cost]

  M[, efficiency := fifelse(
    cumulative_additional_demand > 0,
    cumulative_additional_cost / cumulative_additional_demand,
    NA_real_
  )]

  M[, .(
    run_id,
    year,
    centrality_type,
    intervention,
    additional_annual_demand,
    additional_annual_cost,
    efficiency
  )]
}

compute_diffs <- function(dt) {

  long <- melt(
    as.data.table(dt),
    id.vars = c("run_id", "year", "centrality_type", "intervention"),
    measure.vars = c("additional_annual_demand", "additional_annual_cost", "efficiency"),
    variable.name = "metric",
    value.name = "value"
  )

  H <- long[intervention == "High Centrality"]
  R <- long[intervention == "Random"]

  setkey(H, run_id, year, centrality_type, metric)
  setkey(R, run_id, year, centrality_type, metric)

  H[R, .(
    run_id,
    year,
    centrality_type,
    metric,
    contrast = "diff_HvsR",
    diff = value - i.value
  )]
}

summ_iqr <- function(dt) {
  dt[, .(
    med_diff = median(diff, na.rm = TRUE),
    p25_diff = quantile(diff, 0.25, na.rm = TRUE, type = 1),
    p75_diff = quantile(diff, 0.75, na.rm = TRUE, type = 1)
  ), by = .(year, centrality_type, metric, contrast)]
}

make_demand_cost_panel <- function(df_demand, df_cost, title, ylab_left, ylab_right,
                                   demand_ylim, scale_factor) {

  df_demand <- copy(df_demand)
  df_cost   <- copy(df_cost)

  df_demand[, centrality_type := factor(
    centrality_type,
    levels = c("Degree", "Betweenness")
  )]

  df_cost[, centrality_type := factor(
    centrality_type,
    levels = c("Degree", "Betweenness")
  )]

  df_cost[, `:=`(
    med_diff_scaled = med_diff * scale_factor,
    p25_diff_scaled = p25_diff * scale_factor,
    p75_diff_scaled = p75_diff * scale_factor
  )]

  ggplot() +
    geom_hline(
      yintercept = 0,
      linewidth = 0.7,
      colour = "black"
    ) +
    geom_errorbar(
      data = df_demand,
      aes(x = year, ymin = p25_diff, ymax = p75_diff, colour = "Demand"),
      width = 0,
      linewidth = 0.75,
      alpha = 0.15
    ) +
    geom_line(
      data = df_demand,
      aes(x = year, y = med_diff, colour = "Demand"),
      linewidth = 1.15
    ) +
    geom_errorbar(
      data = df_cost,
      aes(x = year, ymin = p25_diff_scaled, ymax = p75_diff_scaled, colour = "Policy cost"),
      width = 0,
      linewidth = 0.75,
      alpha = 0.15
    ) +
    geom_line(
      data = df_cost,
      aes(x = year, y = med_diff_scaled, colour = "Policy cost"),
      linewidth = 0.9,
      linetype = "22",
      alpha = 0.95
    ) +
    scale_colour_manual(
      values = c(
        "Demand" = "#2C7FB8",
        "Policy cost" = "maroon"
      ),
      name = NULL
    ) +
    scale_y_continuous(
      limits = demand_ylim,
      name = ylab_left,
      sec.axis = sec_axis(
        ~ . / scale_factor,
        name = ylab_right
      )
    ) +
    scale_x_continuous(
      breaks = seq(2020, max(df_demand$year, na.rm = TRUE), by = 20),
      expand = expansion(mult = c(0.01, 0.12))
    ) +
    facet_wrap(
      ~ centrality_type,
      nrow = 1,
      labeller = as_labeller(c(
        Degree = expression(H[2]~"valleys"),
        Betweenness = expression(H[2]~"corridors")
      ))
    ) +
    labs(
      title = title,
      x = "Year"
    ) +
    plot_theme +
    theme(
      legend.position = "bottom",
      legend.text = element_text(size = 18),
      axis.title.x = element_text(size = 20),
      axis.title.y = element_text(size = 20),
      axis.title.y.right = element_text(size = 20, color = "grey30"),
      axis.text.y.right  = element_text(size = 20, color = "grey30")
    )
}

make_efficiency_panel <- function(df_eff, title, ylab, ylim_use) {

  df_eff <- copy(df_eff)

  df_eff[, centrality_type := factor(
    centrality_type,
    levels = c("Degree", "Betweenness")
  )]

  ggplot(df_eff, aes(x = year, y = med_diff)) +
    geom_errorbar(
      aes(ymin = p25_diff, ymax = p75_diff),
      width = 0,
      linewidth = 0.75,
      alpha = 0.15,
      color = "#2C7FB8"
    ) +
    geom_line(
      linewidth = 1.15,
      color = "#2C7FB8"
    ) +
    geom_hline(
      yintercept = 0,
      linewidth = 0.7,
      colour = "black"
    ) +
    scale_y_continuous(limits = ylim_use) +
    scale_x_continuous(
      breaks = seq(2020, max(df_eff$year, na.rm = TRUE), by = 20),
      expand = expansion(mult = c(0.01, 0.12))
    ) +
    facet_wrap(
      ~ centrality_type,
      nrow = 1,
      labeller = as_labeller(c(
        Degree = expression(H[2]~"valleys"),
        Betweenness = expression(H[2]~"corridors")
      ))
    ) +
    labs(
      title = title,
      x = "Year",
      y = ylab
    ) +
    plot_theme +
    theme(
      axis.title.x = element_text(size = 20),
      axis.title.y = element_text(size = 20)
    )
}

runs_main <- extract_runs(results_all_dist)
diffs_main <- compute_diffs(runs_main)
summary_main <- summ_iqr(diffs_main)

summary_main[, centrality_type := factor(
  centrality_type,
  levels = c("Degree", "Betweenness")
)]

demand_diff_main <- summary_main[metric == "additional_annual_demand" & contrast == "diff_HvsR"]
cost_diff_main   <- summary_main[metric == "additional_annual_cost" & contrast == "diff_HvsR"]
eff_diff_main    <- summary_main[metric == "efficiency" & contrast == "diff_HvsR"]

runs_auction <- extract_runs(results_all_dist_auction)
diffs_auction <- compute_diffs(runs_auction)
summary_auction <- summ_iqr(diffs_auction)

summary_auction[, centrality_type := factor(
  centrality_type,
  levels = c("Degree", "Betweenness")
)]

demand_diff_auction <- summary_auction[metric == "additional_annual_demand" & contrast == "diff_HvsR"]
cost_diff_auction   <- summary_auction[metric == "additional_annual_cost" & contrast == "diff_HvsR"]
eff_diff_auction    <- summary_auction[metric == "efficiency" & contrast == "diff_HvsR"]

demand_max_all <- max(
  abs(c(
    demand_diff_main$p25_diff,
    demand_diff_main$p75_diff,
    demand_diff_auction$p25_diff,
    demand_diff_auction$p75_diff
  )),
  na.rm = TRUE
)

cost_max_all <- max(
  abs(c(
    cost_diff_main$p25_diff,
    cost_diff_main$p75_diff,
    cost_diff_auction$p25_diff,
    cost_diff_auction$p75_diff
  )),
  na.rm = TRUE
)

shared_demand_ylim <- range(
  c(
    demand_diff_main$p25_diff,
    demand_diff_main$p75_diff,
    demand_diff_auction$p25_diff,
    demand_diff_auction$p75_diff
  ),
  na.rm = TRUE
)
buffer <- 0.05 * diff(shared_demand_ylim)
shared_demand_ylim <- shared_demand_ylim + c(-buffer, buffer)

shared_scale_factor <- if (is.finite(cost_max_all) && cost_max_all > 0) {
  demand_max_all / cost_max_all
} else {
  1
}

eff_range_all <- range(
  c(
    eff_diff_main$p25_diff,
    eff_diff_main$p75_diff,
    eff_diff_auction$p25_diff,
    eff_diff_auction$p75_diff
  ),
  na.rm = TRUE
)

eff_ylim_all <- c(-max(abs(eff_range_all)), max(abs(eff_range_all)))

p_a <- make_demand_cost_panel(
  demand_diff_main,
  cost_diff_main,
  "'Controlled' CfD setting",
  expression(Delta*" Mt"),
  expression(Delta*" Policy cost"),
  shared_demand_ylim,
  shared_scale_factor
)

p_b <- make_efficiency_panel(
  eff_diff_main,
  "",
  expression(Delta*" EUR/kg"),
  eff_ylim_all
)

p_c <- make_demand_cost_panel(
  demand_diff_auction,
  cost_diff_auction,
  "EHB-style setting",
  expression(Delta*" Mt"),
  expression(Delta*" Policy cost"),
  shared_demand_ylim,
  shared_scale_factor
)

p_d <- make_efficiency_panel(
  eff_diff_auction,
  "",
  expression(Delta*" EUR/kg"),
  eff_ylim_all
)

options(repr.plot.width = 20, repr.plot.height = 18, repr.plot.res = 600)

figure6 <- (
  p_a + p_c
) / (
  p_b + p_d
)

figure6 <- figure6 +
  plot_annotation(tag_levels = "a")

print(figure6)

ggsave(
  "figure6.pdf",
  figure6,
  device = cairo_pdf,
  width = 20,
  height = 18,
  dpi = 600
)

In [ ]:
# EXPORT FIGURE 6 SOURCE DATA TO XLSX

# Folder / filename
out_file <- "figure6_source_data.xlsx"


# Helper: prepare run-level data used in Figure 6

prepare_run_source <- function(results_obj, policy_setting) {

  runs <- extract_runs(results_obj)

  runs %>%
    as_tibble() %>%
    mutate(
      policy_setting = policy_setting,
      centrality_type = as.character(centrality_type),
      intervention = as.character(intervention)
    ) %>%
    select(
      policy_setting,
      run_id,
      year,
      centrality_type,
      intervention,
      additional_annual_demand,
      additional_annual_cost,
      efficiency
    ) %>%
    arrange(
      centrality_type,
      intervention,
      run_id,
      year
    )
}


# Helper: prepare run-level High Centrality versus Random contrasts

prepare_contrast_source <- function(results_obj, policy_setting) {

  runs <- extract_runs(results_obj)
  contrasts <- compute_diffs(runs)

  contrasts %>%
    as_tibble() %>%
    mutate(
      policy_setting = policy_setting,
      centrality_type = as.character(centrality_type),
      metric = recode(
        as.character(metric),
        additional_annual_demand = "Additional annual demand",
        additional_annual_cost = "Additional annual policy cost",
        efficiency = "Policy cost per additional unit of demand"
      ),
      contrast = "High Centrality minus Random"
    ) %>%
    select(
      policy_setting,
      run_id,
      year,
      centrality_type,
      metric,
      contrast,
      diff
    ) %>%
    arrange(
      centrality_type,
      metric,
      run_id,
      year
    )
}

# Helper: prepare median and interquartile-range plotting data

prepare_summary_source <- function(results_obj, policy_setting) {

  runs <- extract_runs(results_obj)
  contrasts <- compute_diffs(runs)
  summary_data <- summ_iqr(contrasts)

  summary_data %>%
    as_tibble() %>%
    mutate(
      policy_setting = policy_setting,
      centrality_type = as.character(centrality_type),
      metric_code = as.character(metric),
      metric = recode(
        metric_code,
        additional_annual_demand = "Additional annual demand",
        additional_annual_cost = "Additional annual policy cost",
        efficiency = "Policy cost per additional unit of demand"
      ),
      contrast = "High Centrality minus Random",
      summary_statistic = "Median and interquartile range across simulation runs"
    ) %>%
    select(
      policy_setting,
      year,
      centrality_type,
      metric_code,
      metric,
      contrast,
      med_diff,
      p25_diff,
      p75_diff,
      summary_statistic
    ) %>%
    arrange(
      centrality_type,
      metric_code,
      year
    )
}


# Prepare run-level source data

runs_controlled <- prepare_run_source(
  results_obj = results_all_dist,
  policy_setting = "Controlled CfD setting"
)

runs_ehb <- prepare_run_source(
  results_obj = results_all_dist_auction,
  policy_setting = "EHB-style setting"
)

all_run_data <- bind_rows(
  runs_controlled,
  runs_ehb
)

# Prepare run-level contrast data

contrasts_controlled <- prepare_contrast_source(
  results_obj = results_all_dist,
  policy_setting = "Controlled CfD setting"
)

contrasts_ehb <- prepare_contrast_source(
  results_obj = results_all_dist_auction,
  policy_setting = "EHB-style setting"
)

all_contrast_data <- bind_rows(
  contrasts_controlled,
  contrasts_ehb
)


# Prepare summary data plotted in Figure 6

summary_controlled <- prepare_summary_source(
  results_obj = results_all_dist,
  policy_setting = "Controlled CfD setting"
)

summary_ehb <- prepare_summary_source(
  results_obj = results_all_dist_auction,
  policy_setting = "EHB-style setting"
)

all_summary_data <- bind_rows(
  summary_controlled,
  summary_ehb
)


# Prepare panel-specific plotting data

panel_a_demand <- summary_controlled %>%
  filter(metric_code == "additional_annual_demand") %>%
  transmute(
    panel = "a",
    policy_setting,
    centrality_type,
    year,
    metric,
    median = med_diff,
    percentile_25 = p25_diff,
    percentile_75 = p75_diff,
    plotted_value = med_diff,
    plotted_percentile_25 = p25_diff,
    plotted_percentile_75 = p75_diff,
    axis = "Left axis",
    line_type = "Solid"
  )

panel_a_cost <- summary_controlled %>%
  filter(metric_code == "additional_annual_cost") %>%
  transmute(
    panel = "a",
    policy_setting,
    centrality_type,
    year,
    metric,
    median = med_diff,
    percentile_25 = p25_diff,
    percentile_75 = p75_diff,
    plotted_value = med_diff * shared_scale_factor,
    plotted_percentile_25 = p25_diff * shared_scale_factor,
    plotted_percentile_75 = p75_diff * shared_scale_factor,
    axis = "Right axis",
    line_type = "Dashed"
  )

panel_b_efficiency <- summary_controlled %>%
  filter(metric_code == "efficiency") %>%
  transmute(
    panel = "b",
    policy_setting,
    centrality_type,
    year,
    metric,
    median = med_diff,
    percentile_25 = p25_diff,
    percentile_75 = p75_diff,
    plotted_value = med_diff,
    plotted_percentile_25 = p25_diff,
    plotted_percentile_75 = p75_diff,
    axis = "Left axis",
    line_type = "Solid"
  )

panel_c_demand <- summary_ehb %>%
  filter(metric_code == "additional_annual_demand") %>%
  transmute(
    panel = "c",
    policy_setting,
    centrality_type,
    year,
    metric,
    median = med_diff,
    percentile_25 = p25_diff,
    percentile_75 = p75_diff,
    plotted_value = med_diff,
    plotted_percentile_25 = p25_diff,
    plotted_percentile_75 = p75_diff,
    axis = "Left axis",
    line_type = "Solid"
  )

panel_c_cost <- summary_ehb %>%
  filter(metric_code == "additional_annual_cost") %>%
  transmute(
    panel = "c",
    policy_setting,
    centrality_type,
    year,
    metric,
    median = med_diff,
    percentile_25 = p25_diff,
    percentile_75 = p75_diff,
    plotted_value = med_diff * shared_scale_factor,
    plotted_percentile_25 = p25_diff * shared_scale_factor,
    plotted_percentile_75 = p75_diff * shared_scale_factor,
    axis = "Right axis",
    line_type = "Dashed"
  )

panel_d_efficiency <- summary_ehb %>%
  filter(metric_code == "efficiency") %>%
  transmute(
    panel = "d",
    policy_setting,
    centrality_type,
    year,
    metric,
    median = med_diff,
    percentile_25 = p25_diff,
    percentile_75 = p75_diff,
    plotted_value = med_diff,
    plotted_percentile_25 = p25_diff,
    plotted_percentile_75 = p75_diff,
    axis = "Left axis",
    line_type = "Solid"
  )

panel_a_plot_data <- bind_rows(
  panel_a_demand,
  panel_a_cost
) %>%
  arrange(
    centrality_type,
    metric,
    year
  )

panel_c_plot_data <- bind_rows(
  panel_c_demand,
  panel_c_cost
) %>%
  arrange(
    centrality_type,
    metric,
    year
  )


# Prepare values defining shared plot scales

plot_scale_metadata <- tibble(
  item = c(
    "Demand and policy-cost left-axis minimum",
    "Demand and policy-cost left-axis maximum",
    "Policy-cost scaling factor",
    "Efficiency-axis minimum",
    "Efficiency-axis maximum",
    "Demand and policy-cost x-axis minimum",
    "Demand and policy-cost x-axis maximum",
    "Efficiency x-axis minimum",
    "Efficiency x-axis maximum"
  ),
  value = c(
    shared_demand_ylim[1],
    shared_demand_ylim[2],
    shared_scale_factor,
    eff_ylim_all[1],
    eff_ylim_all[2],
    min(
      panel_a_plot_data$year,
      panel_c_plot_data$year,
      na.rm = TRUE
    ),
    max(
      panel_a_plot_data$year,
      panel_c_plot_data$year,
      na.rm = TRUE
    ),
    min(
      panel_b_efficiency$year,
      panel_d_efficiency$year,
      na.rm = TRUE
    ),
    max(
      panel_b_efficiency$year,
      panel_d_efficiency$year,
      na.rm = TRUE
    )
  )
)


# Figure metadata

figure_metadata <- tibble(
  item = c(
    "Figure",
    "Description",
    "Comparison",
    "Policy settings",
    "Centrality measures",
    "Centrality display labels",
    "Summary statistic",
    "Uncertainty interval",
    "Demand unit",
    "Policy-cost unit",
    "Efficiency unit",
    "Efficiency calculation",
    "Policy-cost axis transformation",
    "Simulation-run identifier",
    "Baseline treatment"
  ),
  value = c(
    "Figure 6",
    paste(
      "Differences in additional hydrogen demand, policy cost and",
      "policy efficiency between High Centrality targeting and",
      "Random targeting under controlled CfD and EHB-style settings."
    ),
    "High Centrality minus Random",
    "Controlled CfD setting; EHB-style setting",
    "Degree centrality; Betweenness centrality",
    "H2 valleys; H2 corridors",
    "Median across simulation runs",
    "25th to 75th percentiles across simulation runs",
    "Mt of hydrogen",
    "Original policy-cost unit generated by the simulation",
    "EUR per kg of additional hydrogen demand",
    paste(
      "Cumulative additional policy cost divided by cumulative",
      "additional hydrogen demand relative to the baseline."
    ),
    paste(
      "Policy-cost values are multiplied by shared_scale_factor",
      "for plotting against the demand axis. The secondary axis",
      "reverses this transformation."
    ),
    "run_id identifies individual simulation runs",
    paste(
      "Additional demand and policy cost are calculated relative",
      "to the matching baseline simulation for the same run,",
      "centrality type and year."
    )
  )
)



# Panel descriptions

panel_metadata <- tibble(
  panel = c(
    "a",
    "b",
    "c",
    "d"
  ),
  policy_setting = c(
    "Controlled CfD setting",
    "Controlled CfD setting",
    "EHB-style setting",
    "EHB-style setting"
  ),
  outcome = c(
    "Additional annual demand and additional annual policy cost",
    "Policy cost per additional unit of demand",
    "Additional annual demand and additional annual policy cost",
    "Policy cost per additional unit of demand"
  ),
  contrast = "High Centrality minus Random",
  centrality_facets = "Degree centrality; Betweenness centrality",
  displayed_facet_labels = "H2 valleys; H2 corridors",
  plotted_summary = "Median",
  interval = "25th to 75th percentiles"
)



# Metric definitions

metric_definitions <- tibble(
  variable = c(
    "additional_annual_demand",
    "additional_annual_cost",
    "efficiency",
    "diff",
    "med_diff",
    "p25_diff",
    "p75_diff",
    "plotted_value",
    "plotted_percentile_25",
    "plotted_percentile_75"
  ),
  definition = c(
    paste(
      "Annual hydrogen demand under the intervention minus annual",
      "hydrogen demand under the matching baseline."
    ),
    paste(
      "Annual policy cost under the intervention minus annual",
      "policy cost under the matching baseline."
    ),
    paste(
      "Cumulative additional policy cost divided by cumulative",
      "additional hydrogen demand."
    ),
    paste(
      "Value for High Centrality targeting minus the corresponding",
      "value for Random targeting."
    ),
    "Median of diff across simulation runs.",
    "25th percentile of diff across simulation runs.",
    "75th percentile of diff across simulation runs.",
    paste(
      "Value used on the primary plotting scale. Policy-cost values",
      "are multiplied by shared_scale_factor."
    ),
    paste(
      "25th percentile used on the primary plotting scale.",
      "Policy-cost values are multiplied by shared_scale_factor."
    ),
    paste(
      "75th percentile used on the primary plotting scale.",
      "Policy-cost values are multiplied by shared_scale_factor."
    )
  )
)


# Intervention definitions

intervention_definitions <- tibble(
  intervention = c(
    "Baseline",
    "Random",
    "High Centrality",
    "Low Centrality"
  ),
  definition = c(
    "No targeting intervention.",
    "Random or cost-based comparison intervention used in Figure 6.",
    "Intervention targeted towards high-centrality sites.",
    paste(
      "Intervention targeted towards low-centrality sites.",
      "Extracted by the source function but not displayed in Figure 6."
    )
  )
)

# Source sheets

source_sheets <- list(
  "README" = figure_metadata,

  "Panel_descriptions" = panel_metadata,

  "Metric_definitions" = metric_definitions,

  "Interventions" = intervention_definitions,

  "Plot_scales" = plot_scale_metadata,

  "Panel_a_demand_cost" = panel_a_plot_data,

  "Panel_b_efficiency" = panel_b_efficiency,

  "Panel_c_demand_cost" = panel_c_plot_data,

  "Panel_d_efficiency" = panel_d_efficiency,

  "Summary_all_panels" = all_summary_data,

  "Run_contrasts_all" = all_contrast_data,

  "Run_data_controlled" = runs_controlled,

  "Run_data_EHB" = runs_ehb
)

# Write workbook
wb <- createWorkbook()

walk(names(source_sheets), function(sheet_name) {

  sheet_data <- source_sheets[[sheet_name]]

  addWorksheet(
    wb,
    sheetName = sheet_name
  )

  writeData(
    wb,
    sheet = sheet_name,
    x = sheet_data
  )

  freezePane(
    wb,
    sheet = sheet_name,
    firstRow = TRUE
  )

  if (ncol(sheet_data) > 0) {

    addFilter(
      wb,
      sheet = sheet_name,
      row = 1,
      cols = seq_len(ncol(sheet_data))
    )

    setColWidths(
      wb,
      sheet = sheet_name,
      cols = seq_len(ncol(sheet_data)),
      widths = "auto"
    )
  }
})

# Save workbook

saveWorkbook(
  wb,
  out_file,
  overwrite = TRUE
)

In [ ]:
# METRICS FOR TEXT - CFD / CONTROLLED SETTING
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

extract_runs <- function(results_obj) {

  nm <- names(results_obj)
  out <- vector("list", length(nm))

  for (idx in seq_along(nm)) {

    name <- nm[[idx]]
    runs <- results_obj[[name]]

    dt_name <- rbindlist(
      lapply(seq_along(runs), function(i) {
        s <- runs[[i]]
        data.table(
          run_id        = i,
          year          = s$year,
          annual_demand = s$annual_demand / 1e3,
          annual_cost   = s$annual_cost,
          scenario_raw  = name
        )
      })
    )

    out[[idx]] <- dt_name
  }

  DT <- rbindlist(out)

  DT[, centrality_type := fifelse(
    grepl("betweenness", scenario_raw), "Betweenness",
    "Degree"
  )]

  DT[, intervention := fifelse(
    grepl("_high", scenario_raw), "High Centrality",
    fifelse(grepl("_low", scenario_raw), "Low Centrality",
    fifelse(grepl("_random|_cost", scenario_raw), "Random",
    "Baseline"))
  )]

  setorder(DT, run_id, centrality_type, intervention, year)

  DT[, cumulative_demand := cumsum(annual_demand),
     by = .(run_id, centrality_type, intervention)]
  DT[, cumulative_cost := cumsum(annual_cost),
     by = .(run_id, centrality_type, intervention)]

  baseDT <- DT[
    intervention == "Baseline",
    .(
      run_id,
      centrality_type,
      year,
      base_annual_demand = annual_demand,
      base_annual_cost   = annual_cost,
      base_cum_demand    = cumulative_demand,
      base_cum_cost      = cumulative_cost
    )
  ]

  M <- merge(
    DT[intervention != "Baseline"],
    baseDT,
    by = c("run_id", "centrality_type", "year"),
    all.x = TRUE
  )

  M[, additional_annual_demand := annual_demand - base_annual_demand]
  M[, additional_annual_cost   := annual_cost - base_annual_cost]

  M[, cumulative_additional_demand := cumulative_demand - base_cum_demand]
  M[, cumulative_additional_cost   := cumulative_cost - base_cum_cost]

  M[, efficiency := fifelse(
    cumulative_additional_demand > 0,
    cumulative_additional_cost / cumulative_additional_demand,
    NA_real_
  )]

  M[, .(
    run_id,
    year,
    centrality_type,
    intervention,
    annual_demand,
    annual_cost,
    additional_annual_demand,
    additional_annual_cost,
    efficiency
  )]
}

summarize_selected_years <- function(dt, years_keep = c(2030, 2050, 2100)) {
  dt[
    year %in% years_keep,
    .(
      median_abs_diff = median(abs_diff, na.rm = TRUE),
      median_pct_diff = median(pct_diff, na.rm = TRUE)
    ),
    by = .(year, centrality_type, comparison)
  ][order(year, centrality_type, comparison)]
}

compute_demand_diffs <- function(dt) {

  dt <- as.data.table(dt)

  H <- dt[intervention == "High Centrality"]
  R <- dt[intervention == "Random"]
  L <- dt[intervention == "Low Centrality"]

  setkey(H, run_id, year, centrality_type)
  setkey(R, run_id, year, centrality_type)
  setkey(L, run_id, year, centrality_type)

  diff_HR <- H[R][, .(
    run_id,
    year,
    centrality_type,
    comparison = "High vs Random",
    abs_diff   = annual_demand - i.annual_demand,
    pct_diff   = fifelse(!is.na(i.annual_demand) & i.annual_demand != 0,
                         100 * (annual_demand - i.annual_demand) / i.annual_demand,
                         NA_real_)
  )]

  diff_HL <- H[L][, .(
    run_id,
    year,
    centrality_type,
    comparison = "High vs Low",
    abs_diff   = annual_demand - i.annual_demand,
    pct_diff   = fifelse(!is.na(i.annual_demand) & i.annual_demand != 0,
                         100 * (annual_demand - i.annual_demand) / i.annual_demand,
                         NA_real_)
  )]

  rbindlist(list(diff_HR, diff_HL), use.names = TRUE)
}

compute_efficiency_diffs <- function(dt) {

  dt <- as.data.table(dt)

  H <- dt[intervention == "High Centrality"]
  R <- dt[intervention == "Random"]
  L <- dt[intervention == "Low Centrality"]

  setkey(H, run_id, year, centrality_type)
  setkey(R, run_id, year, centrality_type)
  setkey(L, run_id, year, centrality_type)

  diff_HR <- H[R][, .(
    run_id,
    year,
    centrality_type,
    comparison = "High vs Random",
    abs_diff   = efficiency - i.efficiency,
    pct_diff   = fifelse(!is.na(i.efficiency) & i.efficiency != 0,
                         100 * (efficiency - i.efficiency) / i.efficiency,
                         NA_real_)
  )]

  diff_HL <- H[L][, .(
    run_id,
    year,
    centrality_type,
    comparison = "High vs Low",
    abs_diff   = efficiency - i.efficiency,
    pct_diff   = fifelse(!is.na(i.efficiency) & i.efficiency != 0,
                         100 * (efficiency - i.efficiency) / i.efficiency,
                         NA_real_)
  )]

  rbindlist(list(diff_HR, diff_HL), use.names = TRUE)
}

runs_cfd <- extract_runs(results_all_dist)

demand_diffs_cfd <- compute_demand_diffs(runs_cfd)

demand_summary_cfd <- summarize_selected_years(
  demand_diffs_cfd,
  years_keep = 2100
)

demand_summary_cfd

eff_diffs_cfd <- compute_efficiency_diffs(runs_cfd)

eff_summary_cfd <- summarize_selected_years(
  eff_diffs_cfd,
  years_keep = 2100
)

eff_summary_cfd

In [ ]:
# METRICS FOR TEXT - AUCTION / EHB SETTING
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

extract_runs <- function(results_obj) {

  nm <- names(results_obj)
  out <- vector("list", length(nm))

  for (idx in seq_along(nm)) {

    name <- nm[[idx]]
    runs <- results_obj[[name]]

    dt_name <- rbindlist(
      lapply(seq_along(runs), function(i) {
        s <- runs[[i]]
        data.table(
          run_id        = i,
          year          = s$year,
          annual_demand = s$annual_demand / 1e3,
          annual_cost   = s$annual_cost,
          scenario_raw  = name
        )
      })
    )

    out[[idx]] <- dt_name
  }

  DT <- rbindlist(out)

  DT[, centrality_type := fifelse(
    grepl("betweenness", scenario_raw), "Betweenness",
    "Degree"
  )]

  DT[, intervention := fifelse(
    grepl("_high", scenario_raw), "High Centrality",
    fifelse(grepl("_low", scenario_raw), "Low Centrality",
    fifelse(grepl("_random|_cost", scenario_raw), "Random",
    "Baseline"))
  )]

  setorder(DT, run_id, centrality_type, intervention, year)

  DT[, cumulative_demand := cumsum(annual_demand),
     by = .(run_id, centrality_type, intervention)]
  DT[, cumulative_cost := cumsum(annual_cost),
     by = .(run_id, centrality_type, intervention)]

  baseDT <- DT[
    intervention == "Baseline",
    .(
      run_id,
      centrality_type,
      year,
      base_annual_demand = annual_demand,
      base_annual_cost   = annual_cost,
      base_cum_demand    = cumulative_demand,
      base_cum_cost      = cumulative_cost
    )
  ]

  M <- merge(
    DT[intervention != "Baseline"],
    baseDT,
    by = c("run_id", "centrality_type", "year"),
    all.x = TRUE
  )

  M[, additional_annual_demand := annual_demand - base_annual_demand]
  M[, additional_annual_cost   := annual_cost - base_annual_cost]

  M[, cumulative_additional_demand := cumulative_demand - base_cum_demand]
  M[, cumulative_additional_cost   := cumulative_cost - base_cum_cost]

  M[, efficiency := fifelse(
    cumulative_additional_demand > 0,
    cumulative_additional_cost / cumulative_additional_demand,
    NA_real_
  )]

  M[, .(
    run_id,
    year,
    centrality_type,
    intervention,
    annual_demand,
    annual_cost,
    additional_annual_demand,
    additional_annual_cost,
    efficiency
  )]
}

summarize_selected_years <- function(dt, years_keep = c(2030, 2050, 2100)) {
  dt[
    year %in% years_keep,
    .(
      median_abs_diff = median(abs_diff, na.rm = TRUE),
      median_pct_diff = median(pct_diff, na.rm = TRUE)
    ),
    by = .(year, centrality_type, comparison)
  ][order(year, centrality_type, comparison)]
}

compute_demand_diffs <- function(dt) {

  dt <- as.data.table(dt)

  H <- dt[intervention == "High Centrality"]
  R <- dt[intervention == "Random"]
  L <- dt[intervention == "Low Centrality"]

  setkey(H, run_id, year, centrality_type)
  setkey(R, run_id, year, centrality_type)
  setkey(L, run_id, year, centrality_type)

  diff_HR <- H[R][, .(
    run_id,
    year,
    centrality_type,
    comparison = "High vs Random",
    abs_diff   = annual_demand - i.annual_demand,
    pct_diff   = fifelse(!is.na(i.annual_demand) & i.annual_demand != 0,
                         100 * (annual_demand - i.annual_demand) / i.annual_demand,
                         NA_real_)
  )]

  diff_HL <- H[L][, .(
    run_id,
    year,
    centrality_type,
    comparison = "High vs Low",
    abs_diff   = annual_demand - i.annual_demand,
    pct_diff   = fifelse(!is.na(i.annual_demand) & i.annual_demand != 0,
                         100 * (annual_demand - i.annual_demand) / i.annual_demand,
                         NA_real_)
  )]

  rbindlist(list(diff_HR, diff_HL), use.names = TRUE)
}

compute_efficiency_diffs <- function(dt) {

  dt <- as.data.table(dt)

  H <- dt[intervention == "High Centrality"]
  R <- dt[intervention == "Random"]
  L <- dt[intervention == "Low Centrality"]

  setkey(H, run_id, year, centrality_type)
  setkey(R, run_id, year, centrality_type)
  setkey(L, run_id, year, centrality_type)

  diff_HR <- H[R][, .(
    run_id,
    year,
    centrality_type,
    comparison = "High vs Random",
    abs_diff   = efficiency - i.efficiency,
    pct_diff   = fifelse(!is.na(i.efficiency) & i.efficiency != 0,
                         100 * (efficiency - i.efficiency) / i.efficiency,
                         NA_real_)
  )]

  diff_HL <- H[L][, .(
    run_id,
    year,
    centrality_type,
    comparison = "High vs Low",
    abs_diff   = efficiency - i.efficiency,
    pct_diff   = fifelse(!is.na(i.efficiency) & i.efficiency != 0,
                         100 * (efficiency - i.efficiency) / i.efficiency,
                         NA_real_)
  )]

  rbindlist(list(diff_HR, diff_HL), use.names = TRUE)
}

runs_auction <- extract_runs(results_all_dist_auction)

demand_diffs_auction <- compute_demand_diffs(runs_auction)

demand_summary_auction <- summarize_selected_years(
  demand_diffs_auction,
  years_keep = 2100
)

demand_summary_auction

eff_diffs_auction <- compute_efficiency_diffs(runs_auction)

eff_summary_auction <- summarize_selected_years(
  eff_diffs_auction,
  years_keep = 2100
)

eff_summary_auction

In [ ]:
# SUPPLEMENTARY FIGURE 14: Expected green H2 and cost effectiveness for demand-side policy interventions targeted on spatial spillover potential: Results for different baseline specifications
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

plot_theme <- theme_minimal(base_size = 20) +
  theme(
    panel.grid   = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    axis.line    = element_line(color = "black"),
    axis.text    = element_text(size = 18),
    strip.text   = element_text(size = 18, face = "bold")
  )

slope_colors <- c(
  "Conservative" = "#E64B35FF",
  "Mean"         = "#4DBBD5FF",
  "Progressive"  = "#008087FF"
)

scenario_list <- list(
  "Mean – Restricted"         = results_all_dist_mean_restricted,
  "Mean – Central"            = results_all_dist,
  "Mean – Extended"           = results_all_dist_mean_extended,
  "Conservative – Restricted" = results_all_dist_cons_restricted,
  "Conservative – Central"    = results_all_dist_cons_central,
  "Conservative – Extended"   = results_all_dist_cons_extended,
  "Progressive – Restricted"  = results_all_dist_progr_restricted,
  "Progressive – Central"     = results_all_dist_progr_central,
  "Progressive – Extended"    = results_all_dist_progr_extended
)

extract_runs <- function(results_obj) {

  nm <- names(results_obj)
  out <- vector("list", length(results_obj))

  for (idx in seq_along(nm)) {

    name <- nm[[idx]]
    runs <- results_obj[[name]]

    dt_name <- rbindlist(
      lapply(seq_along(runs), function(i) {
        s <- runs[[i]]
        data.table(
          run_id        = i,
          year          = s$year,
          annual_demand = s$annual_demand / 1e3,
          annual_cost   = s$annual_cost,
          scenario_raw  = name
        )
      })
    )

    out[[idx]] <- dt_name
  }

  DT <- rbindlist(out)

  DT[, centrality_type := fifelse(
    grepl("betweenness", scenario_raw), "Betweenness",
    "Degree"
  )]

  DT[, intervention := fifelse(
    grepl("_high", scenario_raw), "High Centrality",
    fifelse(
      grepl("_low", scenario_raw), "Low Centrality",
      fifelse(
        grepl("_random|_cost", scenario_raw), "Random",
        "Baseline"
      )
    )
  )]

  setorder(DT, run_id, centrality_type, intervention, year)

  DT[, cumulative_demand := cumsum(annual_demand),
     by = .(run_id, centrality_type, intervention)]
  DT[, cumulative_cost := cumsum(annual_cost),
     by = .(run_id, centrality_type, intervention)]

  baseDT <- DT[
    intervention == "Baseline",
    .(
      run_id,
      centrality_type,
      year,
      base_annual_demand = annual_demand,
      base_annual_cost   = annual_cost,
      base_cum_demand    = cumulative_demand,
      base_cum_cost      = cumulative_cost
    )
  ]

  M <- merge(
    DT[intervention != "Baseline"],
    baseDT,
    by = c("run_id", "centrality_type", "year"),
    all.x = TRUE
  )

  M[, additional_annual_demand := annual_demand - base_annual_demand]
  M[, additional_annual_cost   := annual_cost - base_annual_cost]

  M[, cumulative_additional_demand := cumulative_demand - base_cum_demand]
  M[, cumulative_additional_cost   := cumulative_cost - base_cum_cost]

  M[, efficiency := fifelse(
    cumulative_additional_demand > 0,
    cumulative_additional_cost / cumulative_additional_demand,
    NA_real_
  )]

  M[, .(
    run_id,
    year,
    centrality_type,
    intervention,
    additional_annual_demand,
    additional_annual_cost,
    efficiency
  )]
}

compute_diffs <- function(dt) {

  dt <- as.data.table(dt)

  long <- melt(
    dt,
    id.vars       = c("run_id", "year", "centrality_type", "intervention"),
    measure.vars  = c("additional_annual_demand", "additional_annual_cost", "efficiency"),
    variable.name = "metric",
    value.name    = "value"
  )

  H <- long[intervention == "High Centrality"]
  R <- long[intervention == "Random"]
  L <- long[intervention == "Low Centrality"]

  setkey(H, run_id, year, centrality_type, metric)
  setkey(R, run_id, year, centrality_type, metric)
  setkey(L, run_id, year, centrality_type, metric)

  diff_HR <- H[R, .(
    run_id,
    year,
    centrality_type,
    metric,
    contrast = "High vs Random",
    diff = value - i.value
  )]

  diff_HL <- H[L, .(
    run_id,
    year,
    centrality_type,
    metric,
    contrast = "High vs Low",
    diff = value - i.value
  )]

  rbindlist(list(diff_HR, diff_HL), use.names = TRUE)
}

all_summary <- rbindlist(
  lapply(names(scenario_list), function(nm) {
    dt  <- extract_runs(scenario_list[[nm]])
    dif <- compute_diffs(dt)
    dif[, scenario := nm]
    dif
  }),
  use.names = TRUE
)

all_summary <- all_summary %>%
  mutate(
    slope = case_when(
      grepl("Progressive", scenario)  ~ "Progressive",
      grepl("Conservative", scenario) ~ "Conservative",
      TRUE                            ~ "Mean"
    ),
    group = case_when(
      grepl("Extended", scenario)   ~ "Extended",
      grepl("Restricted", scenario) ~ "Restricted",
      TRUE                          ~ "Central"
    )
  )

all_summary$centrality_type <- factor(
  all_summary$centrality_type,
  levels = c("Degree", "Betweenness")
)

all_summary$group <- factor(
  all_summary$group,
  levels = c("Restricted", "Central", "Extended")
)

summary_diff <- all_summary[
  , .(
    med_diff = median(diff, na.rm = TRUE),
    q25      = quantile(diff, 0.25, na.rm = TRUE, type = 1),
    q75      = quantile(diff, 0.75, na.rm = TRUE, type = 1)
  ),
  by = .(year, centrality_type, group, slope, metric, contrast)
]

make_panel <- function(df_sub, title, ylab) {

  ggplot(df_sub, aes(x = year, group = slope)) +
    geom_ribbon(
      aes(ymin = q25, ymax = q75, fill = slope),
      alpha = 0.15,
      color = NA
    ) +
    geom_line(
      aes(y = med_diff, color = slope),
      linewidth = 1.05
    ) +
    geom_hline(yintercept = 0, color = "black") +
    scale_color_manual(values = slope_colors, name = "Scenario type") +
    scale_fill_manual(values = slope_colors, guide = "none") +
    scale_y_continuous(
      labels = scales::label_number(
        big.mark = ",",
        decimal.mark = ".",
        accuracy = 0.01
      )
    ) +
    facet_grid(
      rows = vars(group),
      cols = vars(centrality_type, contrast),
      scales = "free_y"
    ) +
    labs(title = title, x = "Year", y = ylab) +
    plot_theme
}

p_a <- make_panel(
  summary_diff[metric == "additional_annual_demand"],
  "Δ Additional Green H₂ Demand",
  "Δ Mt"
)

p_b <- make_panel(
  summary_diff[metric == "additional_annual_cost"],
  "Δ Policy Cost",
  "Δ EUR bn"
)

p_c <- make_panel(
  summary_diff[metric == "efficiency"],
  "Δ Policy Cost Effectiveness",
  "Δ EUR/kg"
)

options(repr.plot.width = 18, repr.plot.height = 34, repr.plot.res = 600)

supplementary_figure_15 <- p_a / p_b / p_c +
  plot_annotation(tag_levels = "a")

print(supplementary_figure_15)

ggsave(
  "supplementary-figure-15.pdf",
  supplementary_figure_15,
  device = cairo_pdf,
  width  = 18,
  height = 25,
  units  = "in",
  dpi    = 800
)

In [ ]:
# SUPPLEMENTARY FIGURE 15: Expected green H2 and cost effectiveness for demand-side policy interventions targeted on spatial spillover potential: Results for different baseline specifications (Auction)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

plot_theme <- theme_minimal(base_size = 20) +
  theme(
    panel.grid   = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    axis.line    = element_line(color = "black"),
    axis.text    = element_text(size = 18),
    strip.text   = element_text(size = 18, face = "bold")
  )

slope_colors <- c(
  "Conservative" = "#E64B35FF",
  "Mean"         = "#4DBBD5FF",
  "Progressive"  = "#008087FF"
)

scenario_list <- list(
  "Mean – Restricted"         = results_all_dist_mean_restricted_auction,
  "Mean – Central"            = results_all_dist_auction,
  "Mean – Extended"           = results_all_dist_mean_extended_auction,
  "Conservative – Restricted" = results_all_dist_cons_restricted_auction,
  "Conservative – Central"    = results_all_dist_cons_central_auction,
  "Conservative – Extended"   = results_all_dist_cons_extended_auction,
  "Progressive – Restricted"  = results_all_dist_progr_restricted_auction,
  "Progressive – Central"     = results_all_dist_progr_central_auction,
  "Progressive – Extended"    = results_all_dist_progr_extended_auction
)

extract_runs <- function(results_obj) {

  nm <- names(results_obj)
  out <- vector("list", length(results_obj))

  for (idx in seq_along(nm)) {

    name <- nm[[idx]]
    runs <- results_obj[[name]]

    dt_name <- rbindlist(
      lapply(seq_along(runs), function(i) {
        s <- runs[[i]]
        data.table(
          run_id        = i,
          year          = s$year,
          annual_demand = s$annual_demand / 1e3,
          annual_cost   = s$annual_cost,
          scenario_raw  = name
        )
      })
    )

    out[[idx]] <- dt_name
  }

  DT <- rbindlist(out)

  DT[, centrality_type := fifelse(
    grepl("betweenness", scenario_raw), "Betweenness",
    "Degree"
  )]

  DT[, intervention := fifelse(
    grepl("_high", scenario_raw), "High Centrality",
    fifelse(
      grepl("_low", scenario_raw), "Low Centrality",
      fifelse(
        grepl("_random|_cost", scenario_raw), "Random",
        "Baseline"
      )
    )
  )]

  setorder(DT, run_id, centrality_type, intervention, year)

  DT[, cumulative_demand := cumsum(annual_demand),
     by = .(run_id, centrality_type, intervention)]
  DT[, cumulative_cost := cumsum(annual_cost),
     by = .(run_id, centrality_type, intervention)]

  baseDT <- DT[
    intervention == "Baseline",
    .(
      run_id,
      centrality_type,
      year,
      base_annual_demand = annual_demand,
      base_annual_cost   = annual_cost,
      base_cum_demand    = cumulative_demand,
      base_cum_cost      = cumulative_cost
    )
  ]

  M <- merge(
    DT[intervention != "Baseline"],
    baseDT,
    by = c("run_id", "centrality_type", "year"),
    all.x = TRUE
  )

  M[, additional_annual_demand := annual_demand - base_annual_demand]
  M[, additional_annual_cost   := annual_cost - base_annual_cost]

  M[, cumulative_additional_demand := cumulative_demand - base_cum_demand]
  M[, cumulative_additional_cost   := cumulative_cost - base_cum_cost]

  M[, efficiency := fifelse(
    cumulative_additional_demand > 0,
    cumulative_additional_cost / cumulative_additional_demand,
    NA_real_
  )]

  M[, .(
    run_id,
    year,
    centrality_type,
    intervention,
    additional_annual_demand,
    additional_annual_cost,
    efficiency
  )]
}

compute_diffs <- function(dt) {

  dt <- as.data.table(dt)

  long <- melt(
    dt,
    id.vars       = c("run_id", "year", "centrality_type", "intervention"),
    measure.vars  = c("additional_annual_demand", "additional_annual_cost", "efficiency"),
    variable.name = "metric",
    value.name    = "value"
  )

  H <- long[intervention == "High Centrality"]
  R <- long[intervention == "Random"]
  L <- long[intervention == "Low Centrality"]

  setkey(H, run_id, year, centrality_type, metric)
  setkey(R, run_id, year, centrality_type, metric)
  setkey(L, run_id, year, centrality_type, metric)

  diff_HR <- H[R, .(
    run_id,
    year,
    centrality_type,
    metric,
    contrast = "High vs Random",
    diff = value - i.value
  )]

  diff_HL <- H[L, .(
    run_id,
    year,
    centrality_type,
    metric,
    contrast = "High vs Low",
    diff = value - i.value
  )]

  rbindlist(list(diff_HR, diff_HL), use.names = TRUE)
}

all_summary <- rbindlist(
  lapply(names(scenario_list), function(nm) {
    dt  <- extract_runs(scenario_list[[nm]])
    dif <- compute_diffs(dt)
    dif[, scenario := nm]
    dif
  }),
  use.names = TRUE
)

all_summary <- all_summary %>%
  mutate(
    slope = case_when(
      grepl("Progressive", scenario)  ~ "Progressive",
      grepl("Conservative", scenario) ~ "Conservative",
      TRUE                            ~ "Mean"
    ),
    group = case_when(
      grepl("Extended", scenario)   ~ "Extended",
      grepl("Restricted", scenario) ~ "Restricted",
      TRUE                          ~ "Central"
    )
  )

all_summary$centrality_type <- factor(
  all_summary$centrality_type,
  levels = c("Degree", "Betweenness")
)

all_summary$group <- factor(
  all_summary$group,
  levels = c("Restricted", "Central", "Extended")
)

summary_diff <- all_summary[
  , .(
    med_diff = median(diff, na.rm = TRUE),
    q25      = quantile(diff, 0.25, na.rm = TRUE, type = 1),
    q75      = quantile(diff, 0.75, na.rm = TRUE, type = 1)
  ),
  by = .(year, centrality_type, group, slope, metric, contrast)
]

make_panel <- function(df_sub, title, ylab) {

  ggplot(df_sub, aes(x = year, group = slope)) +
    geom_ribbon(
      aes(ymin = q25, ymax = q75, fill = slope),
      alpha = 0.15,
      color = NA
    ) +
    geom_line(
      aes(y = med_diff, color = slope),
      linewidth = 1.05
    ) +
    geom_hline(yintercept = 0, color = "black") +
    scale_color_manual(values = slope_colors, name = "Scenario type") +
    scale_fill_manual(values = slope_colors, guide = "none") +
    scale_y_continuous(
      labels = scales::label_number(
        big.mark = ",",
        decimal.mark = ".",
        accuracy = 0.01
      )
    ) +
    facet_grid(
      rows = vars(group),
      cols = vars(centrality_type, contrast),
      scales = "free_y"
    ) +
    labs(title = title, x = "Year", y = ylab) +
    plot_theme
}

p_a <- make_panel(
  summary_diff[metric == "additional_annual_demand"],
  "Δ Additional Green H₂ Demand",
  "Δ Mt"
)

p_b <- make_panel(
  summary_diff[metric == "additional_annual_cost"],
  "Δ Policy Cost",
  "Δ EUR bn"
)

p_c <- make_panel(
  summary_diff[metric == "efficiency"],
  "Δ Policy Cost Effectiveness",
  "Δ EUR/kg"
)

options(repr.plot.width = 18, repr.plot.height = 34, repr.plot.res = 600)

supplementary_figure_16 <- p_a / p_b / p_c +
  plot_annotation(tag_levels = "a")

print(supplementary_figure_16)

ggsave(
  "supplementary-figure-16.pdf",
  supplementary_figure_16,
  device = cairo_pdf,
  width  = 18,
  height = 25,
  units  = "in",
  dpi    = 800
)